# Freight Rate Prediction

## 0. Setup
Imports, file paths, and random seed for reproducibility.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [13]:
SEED = 42
DATA_DIR = 'data/'
train_test_file = 'train_test.csv'
dec_chart_file = 'december_chart_inputs.csv'
val_file = 'validation.csv'
val_template_file = 'validation_predictions_template.csv'

In [14]:
df_train_test = pd.read_csv(DATA_DIR+train_test_file, parse_dates=['date'])
df_train_test.head(5)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [15]:
df_val = pd.read_csv(DATA_DIR+val_file, parse_dates=['date'])
df_val.head(5)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541


In [16]:
df_dec_chart = pd.read_csv(DATA_DIR+dec_chart_file, parse_dates=['date'])
df_dec_chart.head(5)

,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN


In [17]:
df_val_template = pd.read_csv(DATA_DIR+val_template_file)
df_val_template.head(5)

,load_id,predicted_rate
0,TE-000001,NaN
1,TE-000002,NaN
2,TE-000003,NaN
3,TE-000004,NaN
4,TE-000005,NaN


## 1. Data Overview
Load the three datasets and check shape, columns, date ranges, missing values, duplicates, and category coverage.


In [18]:
# Section 1 setup: all datasets in one place
datasets = {
    'train_test': df_train_test,
    'validation': df_val,
    'december_chart': df_dec_chart,
    'val_template': df_val_template,
}

### 1.1 Shape and data types

In [19]:
shape_rows = []
for name, df in datasets.items():
    shape_rows.append({'dataset': name, 'rows': df.shape[0], 'columns': df.shape[1]})

pd.DataFrame(shape_rows)

,dataset,rows,columns
0,train_test,48000,14
1,validation,12000,13
2,december_chart,31,7
3,val_template,12000,2


In [20]:
# Data types side by side (NaN = column not present in that dataset)
pd.DataFrame({name: df.dtypes for name, df in datasets.items()})

,train_test,validation,december_chart,val_template
date,datetime64[us],datetime64[us],datetime64[us],NaN
delivery,str,str,str,NaN
delivery_lat,float64,float64,NaN,NaN
delivery_lon,float64,float64,NaN,NaN
distance,float64,float64,int64,NaN
equipment,str,str,str,NaN
load_id,str,str,NaN,str
market_index,float64,float64,NaN,NaN
pickup,str,str,str,NaN
pickup_lat,float64,float64,NaN,NaN


In [21]:
print('In train, not in validation:', set(df_train_test.columns) - set(df_val.columns))
print('In train, not in December:  ', set(df_train_test.columns) - set(df_dec_chart.columns))

In train, not in validation: {'posted_rate'}
In train, not in December:   {'market_index', 'pickup_lon', 'delivery_lat', 'load_id', 'quote_signal', 'posted_rate', 'pickup_lat', 'delivery_lon'}


### 1.2 Date ranges

In [22]:
date_rows = []
for name, df in datasets.items():
    if 'date' in df.columns:
        date_rows.append({
            'dataset': name,
            'min_date': df['date'].min(),
            'max_date': df['date'].max(),
        })

pd.DataFrame(date_rows)

,dataset,min_date,max_date
0,train_test,2025-01-01,2025-10-31
1,validation,2025-11-01,2025-12-31
2,december_chart,2025-12-01,2025-12-31


### 1.3 Missing values

In [23]:
missing_cols = {}
for name, df in datasets.items():
    nulls = df.isnull().sum()
    missing_cols[name] = nulls[nulls > 0]

pd.DataFrame(missing_cols)

,train_test,validation,december_chart,val_template
market_index,374.0,249.0,NaN,NaN
predicted_rate,NaN,NaN,31.0,12000.0
weight,300.0,165.0,NaN,NaN


### 1.4 Duplicates

In [24]:
df_train_test.duplicated().sum()

np.int64(0)

In [25]:
dups_cols = {}
for name, df in datasets.items():
    cols = set(df.columns) - set(['load_id'])
    dups = df.duplicated(cols).sum()
    dups_cols[name] = dups

pd.Series(dups_cols)

train_test            0
validation            0
december_chart        0
val_template      11999
dtype: int64

### 1.5 Categories and coverage

In [26]:
categories = {}
coverage = {}
for name, df in datasets.items():
    categories[name] = df.nunique()

pd.DataFrame(categories)

,train_test,validation,december_chart,val_template
date,304.0,61.0,31.0,NaN
delivery,64.0,72.0,1.0,NaN
delivery_lat,64.0,72.0,NaN,NaN
delivery_lon,63.0,71.0,NaN,NaN
distance,21204.0,9309.0,1.0,NaN
equipment,3.0,3.0,1.0,NaN
load_id,48000.0,12000.0,NaN,12000.0
market_index,32884.0,9520.0,NaN,NaN
pickup,64.0,72.0,1.0,NaN
pickup_lat,64.0,72.0,NaN,NaN


In [27]:
#Coverage shows which cities and lanes in validation and December never appear in train, meaning the model has no history for them
def get_values(df, col):
    """Return a column, or build the lane (pickup -> delivery) on the fly."""
    if col == 'lane':
        return df['pickup'] + ' -> ' + df['delivery']
    return df[col]

coverage_rows = []
for name in ['validation', 'december_chart']:
    df = datasets[name]
    for col in ['pickup', 'delivery', 'equipment', 'lane']:
        train_values = set(get_values(df_train_test, col).unique())
        target_values = get_values(df, col)
        unseen = set(target_values.unique()) - train_values

        coverage_rows.append({
            'dataset': name,
            'column': col,
            'unseen_values': len(unseen),
        })

pd.DataFrame(coverage_rows)

,dataset,column,unseen_values
0,validation,pickup,8
1,validation,delivery,8
2,validation,equipment,0
3,validation,lane,736
4,december_chart,pickup,0
5,december_chart,delivery,0
6,december_chart,equipment,0
7,december_chart,lane,0


### 1.6 Descriptive statistics

In [28]:
df_train_test.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,48000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,2025-05-31 19:53:29.399999,1.083387,2.062468,2373.980682
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,2025-01-01 00:00:00,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,2025-03-18 00:00:00,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,2025-05-31 00:00:00,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,2025-08-15 00:00:00,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,2025-10-31 00:00:00,1.467780,3.610350,25533.000000
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,NaN,0.168091,0.291391,1486.493245


In [29]:
df_val.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,market_index,quote_signal
count,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,11835.000000,12000,11751.000000,12000.000000
mean,35.574216,-90.724880,35.609538,-90.913530,1141.773100,30506.636248,2025-12-01 02:30:28.800000,0.926913,2.051334
min,25.500000,-121.698490,25.500000,-121.698490,70.000000,-47500.000000,2025-11-01 00:00:00,0.723610,1.234760
25%,31.986910,-97.226810,32.009290,-98.400590,556.100000,25493.000000,2025-11-16 00:00:00,0.863880,1.903467
50%,35.294790,-87.528710,35.294790,-87.528710,953.600000,31193.000000,2025-12-01 00:00:00,0.922900,2.051230
75%,39.411040,-82.783790,39.411040,-82.783790,1670.500000,36843.500000,2025-12-16 00:00:00,0.992355,2.200030
max,44.302960,-69.500000,44.302960,-69.500000,3325.700000,47500.000000,2025-12-31 00:00:00,1.098950,2.896350
std,4.366999,13.411638,4.351844,13.624744,732.333797,10572.590450,NaN,0.074109,0.219804


## 2. Exploratory Data Analysis
Distributions of the target and key features, relationships between rate and distance, equipment, lanes, and time.


## 3. Data Cleaning
Handle data-quality issues found in sections 1 and 2. Each decision is documented with its rationale.


## 4. Train/Test Split
Time-based split that mirrors the validation scenario: train on earlier months, test on the most recent ones.


## 5. Feature Engineering
Date features, lane features, and target transformation. Statistics that learn from data are fitted on the training set only.


## 6. Baselines
Simple non-ML references to beat, such as the quote signal multiplied by distance.


## 7. Model Comparison
Model A (all features) vs Model B (December-compatible features), compared against the baselines.


## 8. Final Evaluation
One-time evaluation of the selected model on the held-out test set.


## 9. Final Training and Predictions
Retrain on all development data and generate predictions for validation and December.


## 10. Scoring and December Chart
Run score.py and interpret the December chart.


## 11. Conclusions
Key findings, limitations, and next steps with more time.